
# Обучение ResNet-50 на GTSRB

Этот блокнот демонстрирует полный цикл обучения ResNet-50
на датасете немецких дорожных знаков GTSRB.

In [ ]:
# Импорт библиотек
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Добавляем путь к исходникам
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from configs.resnet50_config import ResNet50Config
from src.data_loader import get_data_loaders
from src.model_resnet50 import create_resnet50
from src.train_resnet50 import Trainer, evaluate_model


In [ ]:

# 1. Проверка GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Загрузка данных
config = ResNet50Config
train_loader, val_loader, test_loader = get_data_loaders(config)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Classes: {config.NUM_CLASSES}")

In [ ]:

# 3. Визуализация данных
def show_batch(dataloader, class_names, n=5):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        # Денормализация
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(class_names[labels[i].item()])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

class_names = config.get_class_names()
show_batch(train_loader, class_names)

# 4. Создание модели
model = create_resnet50(config)
model = model.to(device)

print(f"\nTotal parameters: {model.get_total_params():,}")
print(f"Trainable parameters: {model.get_trainable_params():,}")


# 5. Обучение модели
trainer = Trainer(model, config, device)
history = trainer.train(train_loader, val_loader)

# %%
# 6. Визуализация истории обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# %%
# 7. Оценка на тестовых данных
results, preds, labels = evaluate_model(model, test_loader, config, device)

print("\nTest Results:")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1-Score: {results['f1_score']:.4f}")

# %%
# 8. Визуализация ошибок
def show_misclassified(model, dataloader, class_names, device, n=5):
    model.eval()
    misclassified = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            # Находим ошибки
            mask = preds != labels
            if mask.any():
                for i in range(len(images)):
                    if mask[i] and len(misclassified) < n:
                        misclassified.append({
                            'image': images[i].cpu(),
                            'true_label': labels[i].item(),
                            'pred_label': preds[i].item(),
                            'confidence': torch.softmax(outputs[i], dim=0)[preds[i]].item()
                        })
            if len(misclassified) >= n:
                break
    
    # Визуализация
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i, sample in enumerate(misclassified):
        img = sample['image'].numpy().transpose(1, 2, 0)
        mean = np

In [ ]:
# 3. Визуализация данных
def show_batch(dataloader, class_names, n=5):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        # Денормализация
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(class_names[labels[i].item()])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

class_names = config.get_class_names()
show_batch(train_loader, class_names)

In [ ]:
# 4. Создание модели
model = create_resnet50(config)
model = model.to(device)

print(f"\nTotal parameters: {model.get_total_params():,}")
print(f"Trainable parameters: {model.get_trainable_params():,}")

In [ ]:
# 5. Обучение модели
trainer = Trainer(model, config, device)
history = trainer.train(train_loader, val_loader)

In [ ]:
# 6. Визуализация истории обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 7. Оценка на тестовых данных
results, preds, labels = evaluate_model(model, test_loader, config, device)

print("\nTest Results:")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1-Score: {results['f1_score']:.4f}")

In [ ]:
# 8. Визуализация ошибок
def show_misclassified(model, dataloader, class_names, device, n=5):
    model.eval()
    misclassified = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            # Находим ошибки
            mask = preds != labels
            if mask.any():
                for i in range(len(images)):
                    if mask[i] and len(misclassified) < n:
                        misclassified.append({
                            'image': images[i].cpu(),
                            'true_label': labels[i].item(),
                            'pred_label': preds[i].item(),
                            'confidence': torch.softmax(outputs[i], dim=0)[preds[i]].item()
                        })
            if len(misclassified) >= n:
                break
    
    # Визуализация
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i, sample in enumerate(misclassified):
        img = sample['image'].numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].set_title(f"True: {class_names[sample['true_label']]}\nPred: {class_names[sample['pred_label']]}\nConf: {sample['confidence']:.2f}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

show_misclassified(model, test_loader, class_names, device)

In [ ]:
# 9. Сохранение модели
torch.save(model.state_dict(), config.RUNS_PATH / 'final_model.pth')
print(f"Model saved to: {config.RUNS_PATH / 'final_model.pth'}")